# Cboe 免费样本 NAS 路径机械修复

复用已取得的两份原始 ZIP，零新网页请求。首次上传回执没有 `nas_path` 字段；按仓库归档路径规则做一次上传和限定回读。

In [1]:
import json,hashlib,tarfile,tempfile,subprocess,shlex,zipfile
from pathlib import Path
from src.data.nas_market_data import _settings,nas_api_call
root=Path.cwd();card=root/'research/soxx_option_vendor_sample_preflight_20261023';raw=root/'data/snapshots/soxx_option_vendor_sample_preflight_20261023'
repair=json.loads((card/'nas_path_repair_plan.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
for n,k in [('plan.json','original_plan_sha256'),('sample_receipts.json','sample_receipts_sha256'),('sample_profiles.json','sample_profiles_sha256'),('sample_manifest.json','sample_manifest_sha256')]:assert sha(card/n)==repair[k]
receipts=json.loads((card/'sample_receipts.json').read_text());profiles=json.loads((card/'sample_profiles.json').read_text());manifest=json.loads((card/'sample_manifest.json').read_text())
assert len(receipts)==len(profiles)==2 and all(x['http_status']=='200' and x['curl_exit_code']==0 for x in receipts)
assert set(x['name'] for x in manifest['files'])=={'option_quotes.headers','option_quotes.zip','option_eod.headers','option_eod.zip'}
for item in manifest['files']:
 f=raw/item['name'];assert f.exists() and sha(f)==item['sha256'] and f.stat().st_size==item['bytes']
for name in ['option_quotes.zip','option_eod.zip']:assert zipfile.is_zipfile(raw/name)
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='soxx-option-vendor-repair-') as td:
 archive=Path(td)/'samples.tar.gz'
 with tarfile.open(archive,'w:gz') as t:
  for item in sorted(manifest['files'],key=lambda x:x['name']):t.add(raw/item['name'],arcname=item['name'])
  t.add(card/'sample_manifest.json',arcname='sample_manifest.json')
 digest=sha(archive)
 artifact=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
 assert artifact['sha256']==digest and artifact['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 restored=Path(td)/'restored.tar.gz'
 with restored.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=120)
 assert run.returncode==0 and sha(restored)==digest
 with tarfile.open(restored,'r:gz') as t:
  names={m.name for m in t.getmembers() if m.isfile()};assert names==set(x['name'] for x in manifest['files'])|{'sample_manifest.json'}
  for item in manifest['files']:
   source=t.extractfile(item['name']);assert source is not None and hashlib.sha256(source.read()).hexdigest()==item['sha256']
result={'status':'two_public_Cboe_ZIP_samples_archived_schema_only','plan_sha256':sha(card/'plan.json'),'repair_plan_sha256':sha(card/'nas_path_repair_plan.json'),'GET_attempts':2,'receipts_file':'sample_receipts.json','profiles_file':'sample_profiles.json','manifest_file':'sample_manifest.json','raw_NAS_archive_sha256':digest,'raw_NAS_archive_bytes':artifact['size_bytes'],'NAS_remote_and_scoped_restore_verified':True,'official_product_history_start':'2012-01','SOXX_specific_historical_coverage_proven':False,'2011_Stage1_training_coverage_proven':False,'25_Delta_constant_tenor_feature_qualified':False,'SOXX_market_outcome_reads':0,'model_fits':0,'native_accounts':0,'purchases':0}
(card/'sample_result.json').write_text(json.dumps(result,indent=2)+'\n')
state={'status':'completed','GET_attempts':2,'NAS_archive_sha256':digest,'SOXX_market_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
print({'status':result['status'],'NAS_sha256':digest,'products':[(x['product'],x['raw_zip_bytes']) for x in receipts]})

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'status': 'two_public_Cboe_ZIP_samples_archived_schema_only', 'NAS_sha256': 'ba88ddde28a5878107c6ef48d9a76f0149270b369ed80ba09e9ce4466424cf8f', 'products': [('Option Quotes', 2628937), ('Option EOD Summary', 2952557)]}


## 中文结论

复用先前两份原始 ZIP，零新网页请求；按仓库 NAS 归档路径规则完成归档及限定逐文件回读，SHA-256 `ba88ddde28a5878107c6ef48d9a76f0149270b369ed80ba09e9ce4466424cf8f`。官方产品页历史从 2012-01 开始，免费样本无 SOXX 专属覆盖证明，25 Delta 信号仍未合格。见 `sample_result.json` 与 `conclusion.md`。